In [1]:
# Cell 1: Force TensorFlow to use CPU only (MUST run BEFORE importing tensorflow)
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

print("✅ GPU disabled (CUDA_VISIBLE_DEVICES = -1). TensorFlow will run on CPU.")
print("📍 Working directory:", os.getcwd())


✅ GPU disabled (CUDA_VISIBLE_DEVICES = -1). TensorFlow will run on CPU.
📍 Working directory: /workspace


In [2]:
# Cell 2: Import required libraries + confirm devices
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("✅ All libraries imported successfully")
print("🧠 TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")
cpus = tf.config.list_physical_devices("CPU")
print(f"🖥️ CPUs visible to TF: {len(cpus)}")
print(f"🎮 GPUs visible to TF: {len(gpus)}")
print("✅ Running on CPU only." if len(gpus) == 0 else "⚠️ GPU is visible — restart kernel and run Cell 1 first.")


2026-02-02 13:22:09.104796: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-02 13:22:09.104868: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-02 13:22:09.105843: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


✅ All libraries imported successfully
🧠 TensorFlow version: 2.15.1
🖥️ CPUs visible to TF: 1
🎮 GPUs visible to TF: 0
✅ Running on CPU only.


2026-02-02 13:22:11.749691: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:274] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [3]:
# Cell 3: Configure LOCAL paths for UPPER BODY dataset
import os
from pathlib import Path

LOCAL_ROOT = Path("tight_loose_clothing_augmented/data/upper body clothing categories").resolve()
MODEL_DIR  = Path("models").resolve()
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Local configuration set:")
print("   💾 Local Dataset Path:", LOCAL_ROOT)
print("   💾 Model Output Dir:  ", MODEL_DIR)

if not LOCAL_ROOT.exists():
    raise FileNotFoundError(f"❌ LOCAL_ROOT folder not found: {LOCAL_ROOT}")


✅ Local configuration set:
   💾 Local Dataset Path: /workspace/tight_loose_clothing_augmented/data/upper body clothing categories
   💾 Model Output Dir:   /workspace/models


In [4]:
# Cell 4: Verify local data and count images per category (UPPER BODY)
print("🔍 Verifying UPPER BODY dataset...\n")

valid_ext = (".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp")

categories = sorted([p.name for p in LOCAL_ROOT.iterdir() if p.is_dir()])
print(f"✅ Found {len(categories)} categories:\n")

total_images = 0
for idx, category in enumerate(categories, 1):
    category_path = LOCAL_ROOT / category
    image_files = [f for f in category_path.iterdir() if f.is_file() and f.suffix.lower() in valid_ext]
    image_count = len(image_files)
    total_images += image_count
    print(f"   {idx}. {category}: {image_count} images")

print(f"\n📊 Total images across all categories: {total_images}")

if len(categories) == 0:
    raise ValueError("❌ No category folders found inside LOCAL_ROOT.")
if total_images == 0:
    raise ValueError("❌ No images found. Check file extensions and folder structure.")


🔍 Verifying UPPER BODY dataset...

✅ Found 7 categories:

   1. blouse: 200 images
   2. crop top: 200 images
   3. hoodie: 200 images
   4. shirt: 200 images
   5. sweater: 200 images
   6. tank top: 200 images
   7. tshirt: 200 images

📊 Total images across all categories: 1400


In [5]:
# Cell 5: Configure training parameters
print("⚙️ Configuring training parameters...\n")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 300
NUM_CLASSES = len(categories)

print("📊 Training Configuration:")
print(f"   Image size: {IMG_SIZE}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Max epochs: {EPOCHS}")
print(f"   Number of classes: {NUM_CLASSES}")
print(f"   Training directory: {LOCAL_ROOT}")


⚙️ Configuring training parameters...

📊 Training Configuration:
   Image size: (224, 224)
   Batch size: 32
   Max epochs: 300
   Number of classes: 7
   Training directory: /workspace/tight_loose_clothing_augmented/data/upper body clothing categories


In [6]:
# Cell 6: Create data generators (train/validation split)
print("🎨 Creating data generators...\n")

datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(
    str(LOCAL_ROOT),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True
)

val_generator = datagen.flow_from_directory(
    str(LOCAL_ROOT),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)

print("\n✅ Data generators created successfully")
print(f"📊 Training samples: {train_generator.samples}")
print(f"📊 Validation samples: {val_generator.samples}")

print("\n🏷️ Class indices mapping (IMPORTANT):")
for class_name, class_idx in train_generator.class_indices.items():
    print(f"   {class_idx}: {class_name}")

gen_num_classes = len(train_generator.class_indices)
print(f"\n🔎 Generator classes: {gen_num_classes} | Model NUM_CLASSES: {NUM_CLASSES}")

if gen_num_classes != NUM_CLASSES:
    raise ValueError(f"❌ Class count mismatch: generator={gen_num_classes}, NUM_CLASSES={NUM_CLASSES}")


🎨 Creating data generators...

Found 1120 images belonging to 7 classes.
Found 280 images belonging to 7 classes.

✅ Data generators created successfully
📊 Training samples: 1120
📊 Validation samples: 280

🏷️ Class indices mapping (IMPORTANT):
   0: blouse
   1: crop top
   2: hoodie
   3: shirt
   4: sweater
   5: tank top
   6: tshirt

🔎 Generator classes: 7 | Model NUM_CLASSES: 7


In [7]:
# Cell 7: Quick sanity-check one batch
x_batch, y_batch = next(train_generator)
print("✅ Batch loaded")
print("x_batch:", x_batch.shape, "dtype:", x_batch.dtype, "min/max:", float(x_batch.min()), float(x_batch.max()))
print("y_batch:", y_batch.shape, "dtype:", y_batch.dtype)
print("y_batch sample (first 5 rows):\n", y_batch[:5])


✅ Batch loaded
x_batch: (32, 224, 224, 3) dtype: float32 min/max: 0.0 1.0
y_batch: (32, 7) dtype: float32
y_batch sample (first 5 rows):
 [[0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0.]]


In [8]:
# Cell 8: Load pre-trained MobileNetV2 base model (frozen)
print("🔧 Loading pre-trained MobileNetV2 base model...\n")

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

for layer in base_model.layers:
    layer.trainable = False

print("✅ MobileNetV2 base model loaded")
print(f"   Total layers: {len(base_model.layers)}")
print("   All layers frozen for transfer learning")


🔧 Loading pre-trained MobileNetV2 base model...

✅ MobileNetV2 base model loaded
   Total layers: 154
   All layers frozen for transfer learning


In [9]:
# Cell 9: Build complete model architecture (new classifier head)
print("🏗️ Building complete model architecture...\n")

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

print("✅ Model architecture built successfully\n")
print("📋 Model Summary:")
model.summary()


🏗️ Building complete model architecture...

✅ Model architecture built successfully

📋 Model Summary:
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 Conv1 (Conv2D)              (None, 112, 112, 32)         864       ['input_1[0][0]']             
                                                                                                  
 bn_Conv1 (BatchNormalizati  (None, 112, 112, 32)         128       ['Conv1[0][0]']               
 on)                                                                                              
                                                                                           

In [10]:
# Cell 10: Compile the model (categorical / one-hot labels)
print("⚙️ Compiling the model...\n")

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=[tf.keras.metrics.CategoricalAccuracy(name="accuracy")]
)

print("✅ Model compiled successfully")
print("   Optimizer: Adam (lr=1e-4)")
print("   Loss: categorical_crossentropy")
print("   Metrics: CategoricalAccuracy")


⚙️ Compiling the model...

✅ Model compiled successfully
   Optimizer: Adam (lr=1e-4)
   Loss: categorical_crossentropy
   Metrics: CategoricalAccuracy


In [11]:
# Cell 11: Configure callbacks for training (recommended: monitor val_loss for LR too)
print("🎯 Configuring training callbacks...\n")

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True,
    verbose=1
)

lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",   # <-- better than "loss" for generalization
    factor=0.1,
    patience=10,
    min_lr=1e-7,
    verbose=1
)

print("✅ Callbacks configured:")
print("   - EarlyStopping (monitor=val_loss, patience=20)")
print("   - ReduceLROnPlateau (monitor=val_loss, patience=10, factor=0.1)")


🎯 Configuring training callbacks...

✅ Callbacks configured:
   - EarlyStopping (monitor=val_loss, patience=20)
   - ReduceLROnPlateau (monitor=val_loss, patience=10, factor=0.1)


In [12]:
# Cell 12: Train the model (Phase 1: frozen base)
print("\n" + "="*60)
print("🚀 STARTING MODEL TRAINING (UPPER BODY - PHASE 1: frozen base)")
print("="*60 + "\n")

history1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=[early_stopping, lr_scheduler],
    verbose=1
)

print("\n" + "="*60)
print("✅ PHASE 1 TRAINING COMPLETED")
print("="*60)



🚀 STARTING MODEL TRAINING (UPPER BODY - PHASE 1: frozen base)

Epoch 1/300
35/35 [==============================] - 28s 733ms/step - loss: 2.2468 - accuracy: 0.1679 - val_loss: 1.9254 - val_accuracy: 0.2214 - lr: 1.0000e-04
Epoch 2/300
35/35 [==============================] - 18s 522ms/step - loss: 1.7858 - accuracy: 0.3241 - val_loss: 1.7875 - val_accuracy: 0.2679 - lr: 1.0000e-04
Epoch 3/300
35/35 [==============================] - 18s 510ms/step - loss: 1.5125 - accuracy: 0.4545 - val_loss: 1.6888 - val_accuracy: 0.3786 - lr: 1.0000e-04
Epoch 4/300
35/35 [==============================] - 18s 512ms/step - loss: 1.3435 - accuracy: 0.5241 - val_loss: 1.6144 - val_accuracy: 0.4321 - lr: 1.0000e-04
Epoch 5/300
35/35 [==============================] - 20s 557ms/step - loss: 1.2334 - accuracy: 0.5634 - val_loss: 1.5342 - val_accuracy: 0.4321 - lr: 1.0000e-04
Epoch 6/300
35/35 [==============================] - 19s 529ms/step - loss: 1.1033 - accuracy: 0.6214 - val_loss: 1.4973 - val_accu

In [ ]:
# Cell 13 (Optional): Fine-tune last N layers for better accuracy
print("🔧 Fine-tuning setup...")

FINE_TUNE = True
UNFREEZE_LAST_N = 30   # try 20–60
FINE_TUNE_LR = 1e-5
FINE_TUNE_EPOCHS = 50

if not FINE_TUNE:
    print("⏭️ Fine-tuning skipped (FINE_TUNE=False)")
else:
    # Unfreeze last N layers (keep BatchNorm frozen for stability)
    for layer in base_model.layers[:-UNFREEZE_LAST_N]:
        layer.trainable = False
    for layer in base_model.layers[-UNFREEZE_LAST_N:]:
        if "batch_normalization" in layer.name.lower():
            layer.trainable = False
        else:
            layer.trainable = True

    print(f"✅ Unfroze last {UNFREEZE_LAST_N} layers (BatchNorm kept frozen)")

    model.compile(
        optimizer=Adam(learning_rate=FINE_TUNE_LR),
        loss="categorical_crossentropy",
        metrics=[tf.keras.metrics.CategoricalAccuracy(name="accuracy")]
    )

    print("✅ Re-compiled for fine-tuning")
    print("   Fine-tune LR:", FINE_TUNE_LR)

    print("\n" + "="*60)
    print("🚀 STARTING MODEL TRAINING (UPPER BODY - PHASE 2: fine-tuning)")
    print("="*60 + "\n")

    history2 = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=FINE_TUNE_EPOCHS,
        callbacks=[early_stopping, lr_scheduler],
        verbose=1
    )

    print("\n" + "="*60)
    print("✅ PHASE 2 FINE-TUNING COMPLETED")
    print("="*60)


In [13]:
# Cell 14: Save the trained model (H5 + SavedModel) + timestamped filenames
import time
import tensorflow as tf

timestamp = time.strftime("%Y%m%d_%H%M%S")
H5_PATH = MODEL_DIR / f"upper_body_mobilenetv2_{timestamp}.h5"
SAVEDMODEL_DIR = MODEL_DIR / f"upper_body_mobilenetv2_savedmodel_{timestamp}"

print("\n💾 Saving the trained model...")
print("   H5 path:        ", H5_PATH)
print("   SavedModel dir: ", SAVEDMODEL_DIR)

# Save H5
model.save(str(H5_PATH))
print("✅ Saved H5")

# Save SavedModel (try model.export first; fallback to tf.saved_model.save)
try:
    model.export(str(SAVEDMODEL_DIR))  # TF 2.13+ typically
    print("✅ Saved SavedModel using model.export()")
except Exception as e:
    print("⚠️ model.export() failed, using tf.saved_model.save() instead.")
    print("   Reason:", repr(e))
    tf.saved_model.save(model, str(SAVEDMODEL_DIR))
    print("✅ Saved SavedModel using tf.saved_model.save()")

print("✅ Model saving complete")
print("   H5 exists:", H5_PATH.exists())
print("   SavedModel exists:", SAVEDMODEL_DIR.exists())



💾 Saving the trained model...
   H5 path:         /workspace/models/upper_body_mobilenetv2_20260202_140423.h5
   SavedModel dir:  /workspace/models/upper_body_mobilenetv2_savedmodel_20260202_140423


/usr/local/lib/python3.11/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


✅ Saved H5
INFO:tensorflow:Assets written to: /workspace/models/upper_body_mobilenetv2_savedmodel_20260202_140423/assets


INFO:tensorflow:Assets written to: /workspace/models/upper_body_mobilenetv2_savedmodel_20260202_140423/assets


Saved artifact at '/workspace/models/upper_body_mobilenetv2_savedmodel_20260202_140423'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_1')
Output Type:
  TensorSpec(shape=(None, 7), dtype=tf.float32, name=None)
Captures:
  128678645608528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128678645608720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128678645607952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128678645608144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128678645607760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128678645610448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128678645610256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128678645610832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128678645611024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128678645610640: TensorSpec(shape=(), 

In [14]:
# Cell 15: Save class order EXACTLY as training generator used (CRITICAL for correct JS inference)
import json

idx_to_class = {int(v): k for k, v in train_generator.class_indices.items()}
class_names_in_training_order = [idx_to_class[i] for i in range(len(idx_to_class))]

CLASSES_JSON = MODEL_DIR / f"upper_body_classes_{timestamp}.json"
with open(CLASSES_JSON, "w", encoding="utf-8") as f:
    json.dump(class_names_in_training_order, f, indent=2)

print("✅ Saved classes JSON")
print("   Path:", CLASSES_JSON)
print("   Num classes:", len(class_names_in_training_order))
print("   First 10 classes:", class_names_in_training_order[:10])


✅ Saved classes JSON
   Path: /workspace/models/upper_body_classes_20260202_140423.json
   Num classes: 7
   First 10 classes: ['blouse', 'crop top', 'hoodie', 'shirt', 'sweater', 'tank top', 'tshirt']


In [15]:
# Cell 16: Install tensorflowjs converter (if not installed)
print("📦 Installing / verifying tensorflowjs...")

import sys
!{sys.executable} -m pip install -q tensorflowjs

print("✅ tensorflowjs installed / available")


📦 Installing / verifying tensorflowjs...
✅ tensorflowjs installed / available


In [17]:
# Cell 17: Convert to TensorFlow.js format (model.json + weights)
import tensorflowjs as tfjs

TFJS_DIR = MODEL_DIR / f"tfjs_upper_body_{timestamp}"
TFJS_DIR.mkdir(parents=True, exist_ok=True)

print("🔄 Converting to TensorFlow.js (Layers format)...")
print("   Output folder:", TFJS_DIR)

tfjs.converters.save_keras_model(model, str(TFJS_DIR))

print("✅ TFJS conversion completed")
print("📄 Files created:")
for p in sorted(TFJS_DIR.iterdir()):
    print("  -", p.name)


/usr/local/lib/python3.11/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


🔄 Converting to TensorFlow.js (Layers format)...
   Output folder: /workspace/models/tfjs_upper_body_20260202_140423
✅ TFJS conversion completed
📄 Files created:
  - group1-shard1of3.bin
  - group1-shard2of3.bin
  - group1-shard3of3.bin
  - model.json


In [18]:
# Cell 18: Verify TFJS output
model_json = TFJS_DIR / "model.json"

print("🔎 Verifying TFJS export...")
print("   model.json exists:", model_json.exists())

if not model_json.exists():
    raise FileNotFoundError(f"❌ model.json not found in {TFJS_DIR}")

txt = model_json.read_text(encoding="utf-8")
print("✅ model.json readable")
print("   Preview:", txt[:300].replace("\n", " ") + " ...")


🔎 Verifying TFJS export...
   model.json exists: True
✅ model.json readable
   Preview: {"format": "layers-model", "generatedBy": "keras v2.15.0", "convertedBy": "TensorFlow.js Converter v4.22.0", "modelTopology": {"keras_version": "2.15.0", "backend": "tensorflow", "model_config": {"class_name": "Functional", "config": {"name": "model", "trainable": true, "layers": [{"class_name": "In ...


In [19]:
# Cell 19: Final summary (copy these paths into your JS project)
print("\n" + "="*60)
print("✅ DONE: Upper-body model exported for JavaScript")
print("="*60)

print("🧠 Keras model (.h5):", H5_PATH)
print("🧠 SavedModel dir:   ", SAVEDMODEL_DIR)
print("🌐 TFJS folder:      ", TFJS_DIR)
print("🏷️ Classes JSON:     ", CLASSES_JSON)

print("\n📌 Reminder for JS preprocessing:")
print("   - Resize to 224x224")
print("   - Normalize with /255.0 (same as training)")
print("   - Use class order from the saved JSON (NOT folder sorting)")



✅ DONE: Upper-body model exported for JavaScript
🧠 Keras model (.h5): /workspace/models/upper_body_mobilenetv2_20260202_140423.h5
🧠 SavedModel dir:    /workspace/models/upper_body_mobilenetv2_savedmodel_20260202_140423
🌐 TFJS folder:       /workspace/models/tfjs_upper_body_20260202_140423
🏷️ Classes JSON:      /workspace/models/upper_body_classes_20260202_140423.json

📌 Reminder for JS preprocessing:
   - Resize to 224x224
   - Normalize with /255.0 (same as training)
   - Use class order from the saved JSON (NOT folder sorting)
